# DMRG 03: Models, Backends, and Lanczos

This notebook connects the DMRG workflow to the rest of the tensor-network modules. It shows which MPO builders can be used, how contraction backends are selected, and how the low-level Lanczos solver works on a simple dense operator.


## Supported pieces

The DMRG stack is intentionally modular:

- `quantum_simulation/mpo`: model-specific MPO builders.
- `quantum_simulation/mps.py`: random canonical MPS initialization and center movement.
- `quantum_simulation/eigensolver`: Lanczos ground-state routine used inside local DMRG updates.
- `quantum_simulation/dmrg.py`: sweep schedule, environments, effective Hamiltonian application, and MPS updates.

Any builder that returns a compatible list of rank-4 MPO tensors can be passed to `DMRG`.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_quantum_simulation():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation")


try:
    qs = _import_quantum_simulation()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    qs = _import_quantum_simulation()

print("Loaded quantum_simulation from", Path(qs.__file__).parent)


Loaded quantum_simulation from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation


In [2]:
import torch
from quantum_simulation import (
    DMRG,
    Heisenberg1DMPOBuilder,
    Ising2DMPOBuilder,
    MPS,
    Sweeps,
    lanczos_ground_state,
)

device = "cpu"  # Change to "cuda" after confirming your GPU setup.
dtype = torch.complex128
print(f"Using device={device}, dtype={dtype}")


Using device=cpu, dtype=torch.complex128


## MPO builders available for DMRG

Common builders exported by the package include:

| Builder | Typical use |
| --- | --- |
| `Ising1DMPOBuilder` | 1D transverse-field Ising chains |
| `Heisenberg1DMPOBuilder` | 1D spin-1/2 Heisenberg chains |
| `Ising2DMPOBuilder` | 2D transverse-field Ising lattices mapped to a 1D MPO |
| `Heisenberg2DMPOBuilder` | 2D Heisenberg lattices mapped to a 1D MPO |
| `TJModelMPOBuilder` | 2D t-J model |
| `TTPrimeJModelMPOBuilder` | 2D t-t-prime-J model |
| `T1T2J1J2ModelMPOBuilder` | long-range t1-t2-J1-J2 model |

The DMRG driver only needs the resulting tensor list and the matching physical dimension `chid` for the MPS.


## A small helper for one-sweep examples

The helper below keeps the examples short. It creates a fresh MPS, defines a small sweep schedule, runs DMRG, and returns the final energy.


In [3]:
def run_small_dmrg(mpo, num_sites, physical_dim, *, backend="einsum", seed=0):
    state = MPS(
        Nsites=num_sites,
        chid=physical_dim,
        initial_bond_dim=4,
        device=device,
        dtype=dtype,
        seed=seed,
    )

    sweeps = Sweeps(1)
    sweeps.set_schedule(
        maxdim=[8],
        noise=[0.0],
        krylov_dim=[6],
        cutoff=[1e-12],
        reortho=[True],
        maxreortho=[1],
    )

    solver = DMRG(state, mpo, device=device, dtype=dtype, contract_backend=backend)
    energies, optimized_state = solver(sweeps, dispon=0, maxit=2)
    return energies[-1], optimized_state


## Example 1: Heisenberg chain with different contraction backends

`contract_backend="einsum"` uses direct PyTorch einsum contractions. `contract_backend="auto"` resolves to `ncon` on CPU and `einsum` on GPU. Both paths should produce the same energy for this small example.


In [4]:
num_sites = 4
heisenberg_builder = Heisenberg1DMPOBuilder(
    num_sites=num_sites,
    j_coupling_x=1.0,
    j_coupling_y=1.0,
    j_coupling_z=1.0,
    device=device,
    dtype=dtype,
)
heisenberg_mpo = heisenberg_builder.get_mpo()

for backend in ["einsum", "auto"]:
    energy, state = run_small_dmrg(
        heisenberg_mpo,
        num_sites=num_sites,
        physical_dim=2,
        backend=backend,
        seed=7,
    )
    print(f"backend={backend:6s} final energy = {energy:.12f}")


Initializing MPS as a right-canonical random state...
Initialization complete. Ortho center at site 0.
[DMRG] Contraction backend resolved to 'einsum' on device='cpu'.
backend=einsum final energy = -1.616025403784
Initializing MPS as a right-canonical random state...
Initialization complete. Ortho center at site 0.
[DMRG] Contraction backend resolved to 'ncon' on device='cpu'.
backend=auto   final energy = -1.616025403784


## Example 2: A 2D Ising lattice mapped to a 1D MPO

The 2D builders flatten the lattice into a 1D site order and encode the long-range couplings through the MPO bond dimension. The DMRG call does not change: it still receives an MPO list and an MPS with `Nsites = nx * ny`.


In [5]:
nx, ny = 2, 2
ising_2d_builder = Ising2DMPOBuilder(
    nx=nx,
    ny=ny,
    j_coupling=1.0,
    g_field=0.5,
    device=device,
    dtype=dtype,
)
ising_2d_mpo = ising_2d_builder.get_mpo()

print(f"2D lattice sites: {ising_2d_builder.num_sites}")
print("MPO virtual bond dimensions:")
print([tuple(tensor.shape[:2]) for tensor in ising_2d_mpo])

energy_2d, state_2d = run_small_dmrg(
    ising_2d_mpo,
    num_sites=ising_2d_builder.num_sites,
    physical_dim=2,
    backend="auto",
    seed=11,
)
print(f"2D Ising final energy = {energy_2d:.12f}")


2D lattice sites: 4
MPO virtual bond dimensions:
[(1, 3), (3, 4), (4, 3), (3, 1)]
Initializing MPS as a right-canonical random state...
Initialization complete. Ortho center at site 0.
[DMRG] Contraction backend resolved to 'ncon' on device='cpu'.
2D Ising final energy = -1.306562964876


## Low-level Lanczos example

Inside each two-site DMRG update, the effective Hamiltonian is solved by `lanczos_ground_state`. You can also call the same routine directly with any linear operator. The operator can optionally support an `out=` argument to reuse buffers.


In [6]:
H = torch.tensor(
    [
        [4.0, 1.0, 0.0, 0.5],
        [1.0, 3.0, 0.2, 0.0],
        [0.0, 0.2, 2.0, 0.3],
        [0.5, 0.0, 0.3, 5.0],
    ],
    device=device,
    dtype=torch.float64,
)


def apply_dense_hamiltonian(vec, hamiltonian, out=None):
    result = hamiltonian @ vec
    if out is not None:
        out.copy_(result)
        return out
    return result

initial_vector = torch.rand(H.shape[0], device=device, dtype=H.dtype)
state_vector, lanczos_energy = lanczos_ground_state(
    initial_vector,
    apply_dense_hamiltonian,
    operator_args=(H,),
    num_restarts=6,
    krylov_dim=4,
    pro=True,
    pro_max_reorth=1,
)

exact_energy = torch.linalg.eigvalsh(H)[0].item()
print(f"Lanczos energy: {lanczos_energy:.12f}")
print(f"Exact energy:   {exact_energy:.12f}")
print(f"Absolute error: {abs(lanczos_energy - exact_energy):.3e}")


Lanczos energy: 1.888992593766
Exact energy:   1.888992593766
Absolute error: 2.220e-16


## Takeaways

The same DMRG driver can consume several model builders because the interface is the MPO tensor list. Backend selection changes the contraction implementation, while the Lanczos solver provides the local ground-state search used during every two-site update.
